# Tradr PnL attribution

Company gross:

$$
\mathrm{Gross} = \mathrm{Overnight} + \mathrm{M2M} + \mathrm{TradePnL}
$$

Lifecycle decomposition:

$$
\mathrm{Gross} = \mathrm{Overnight} + \mathrm{MarketRisk} + \mathrm{InternalMarkBasis} + \mathrm{PureExecution}
$$

$$
\mathrm{MarketRisk} = \mathrm{BODInventory} + \mathrm{TransferMark} + \mathrm{PostFillMark}
$$

$$
\mathrm{PureExecution} = \mathrm{MidToTouch} + \mathrm{TouchToExec}
$$

| Term | Formula |
|---|---|
| Overnight | $M Q_0 (X_0 - X_{\mathrm{prevEOD}})$ |
| BOD inventory | $M Q_0 (X_t - X_0)$ |
| Transfer mark | $M \sum q^T (X_t - X_j)$ |
| Post-fill mark | $M \sum q^H (X_t - X_i)$ |
| Internal mark basis | $M \sum q^H (X_i - P)$ |
| Mid → touch | $M \sum q^H (P - T)$ |
| Touch → exec | $M \sum q^H (T - E)$ |

- $M=1000$; trades Buy=+/Sell=−; transfers Sell=+/Buy=−
- $X$ = algo spread curve; $P,T$ = TTMDS mid / touch (ask if buy, bid if sell)
- `residual` should equal `pos_gap_pnl` (inventory not in today's transfers+trades)

In [ ]:
import os
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
from clickhouse_connect import get_client
from dotenv import load_dotenv

load_dotenv()

CONTRACT_SIZE = 1000

BOOK_NAME_TO_ID = {
    'Hedger Spreads': 129,
    'Spreadgr Base': 318,
    'Spreadgr Alt 1': 309,
    'Spreadgr Alt 2': 332,
    'Spreadgr Alt 3': 339,
    'Spreadgr Alt 4': 342,
    'Spreadgr Base 1': 341,
    'Spreadgr Base 2': 337,
    'Spreadgr Base 3': 340,
    'Spreadgr Base 4': 343,
}

client = get_client(
    host=os.getenv('CLICKHOUSE_HOST', '10.20.0.243'),
    port=int(os.getenv('CLICKHOUSE_PORT', '8123')),
    username=os.getenv('CLICKHOUSE_USERNAME', ''),
    password=os.getenv('CLICKHOUSE_PASSWORD', ''),
    database=os.getenv('CLICKHOUSE_DATABASE', 'algo'),
)

In [ ]:
DATE = (datetime.now() - timedelta(days=1)).date()
while DATE.weekday() >= 5:
    DATE -= timedelta(days=1)

HIST_START = DATE - timedelta(days=5)  # overnight needs prior EOD
BOOK = 'Hedger Spreads'  # single-book detail; summary runs all books

print('DATE =', DATE, '| BOOK =', BOOK)

## Data loaders

In [ ]:
def get_spread_positions(book_name: str, start_date, end_date) -> pd.DataFrame:
    """Outrights → consecutive monthly spreads via cumsum across tenors."""
    book_id = BOOK_NAME_TO_ID[book_name]
    raw = client.query_df(f"""
        SELECT timestamp, positions
        FROM algo.position
        WHERE book_id = {book_id}
          AND toDate(timestamp) BETWEEN '{start_date}' AND '{end_date}'
        ORDER BY timestamp
    """)
    if raw.empty:
        return pd.DataFrame()

    outrights = pd.DataFrame(raw['positions'].tolist()).fillna(0)
    outrights.index = pd.DatetimeIndex(raw['timestamp'])
    outrights = outrights.groupby(level=0).sum()
    outrights.columns = pd.to_datetime(outrights.columns)
    outrights = outrights.T.groupby(level=0).sum().T
    outrights = outrights.reindex(sorted(outrights.columns), axis=1)
    months = pd.date_range(outrights.columns.min(), outrights.columns.max(), freq='MS')
    outrights = outrights.reindex(columns=months, fill_value=0)
    return outrights.cumsum(axis=1).groupby(level=0).last().sort_index()


def get_spread_prices(start_date, end_date) -> pd.DataFrame:
    """spread[m] = curve[m] - curve[m+1] from algo.curves Brent."""
    df = client.query_df(f"""
        SELECT timestamp, curve
        FROM algo.curves
        WHERE product = 'Brent_(Fut)'
          AND toDate(timestamp) BETWEEN '{start_date}' AND '{end_date}'
        ORDER BY timestamp
    """)
    prices = pd.json_normalize(df['curve'])
    prices.index = df['timestamp']
    prices.columns = pd.to_datetime(prices.columns)
    prices = prices.reindex(sorted(prices.columns), axis=1)
    spreads = prices.diff(-1, axis=1).iloc[:, :-1]
    return spreads.groupby(level=0).last().sort_index()


def get_trades(book_name, date) -> pd.DataFrame:
    return client.query_df(f"""
        SELECT transaction_timestamp AS timestamp, instrument_key, side, quantity, price
        FROM algo.nexus_trades
        WHERE toDate(transaction_timestamp) = '{date}' AND text_tt = '{book_name}'
        ORDER BY timestamp
    """)


def get_transfers(book_name, date) -> pd.DataFrame:
    return client.query_df(f"""
        SELECT timestamp, start_tenor, end_tenor, side, quantity, price
        FROM algo.nexus_transfers
        WHERE toDate(timestamp) = '{date}' AND source_book = '{book_name}'
        ORDER BY timestamp
    """)


spread_prices = get_spread_prices(HIST_START, DATE)
print('curves', spread_prices.shape)

## Lifecycle decomposition

In [ ]:
def months_between(start, end):
    return list(pd.date_range(start, end, freq='MS')[:-1])


def curve_at(ts, tenors, curves: pd.DataFrame) -> pd.Series:
    i = curves.index.get_indexer([pd.Timestamp(ts)], method='ffill')[0]
    if i < 0:
        return pd.Series(0.0, index=list(tenors))
    row = curves.iloc[i]
    return pd.Series({t: float(row[t]) if t in row.index and pd.notna(row[t]) else 0.0 for t in tenors})


def parse_trade_tenors(instrument_key: str):
    front_s, back_s = str(instrument_key).split()[2].split('-')
    start = pd.to_datetime(front_s, format='%b%y')
    end = pd.to_datetime(back_s, format='%b%y')
    return start, end, months_between(start, end)


def get_ttmds_spread_quotes(asof_date, pairs: list[tuple]) -> pd.DataFrame:
    if not pairs:
        return pd.DataFrame(columns=['timestamp', 'start_tenor', 'end_tenor', 'bid', 'ask', 'mid'])

    pair_sql = ', '.join(
        f"(toDate('{pd.Timestamp(s).date()}'), toDate('{pd.Timestamp(e).date()}'))"
        for s, e in sorted(set((pd.Timestamp(a).date(), pd.Timestamp(b).date()) for a, b in pairs))
    )
    quotes = client.query_df(f"""
        SELECT
            toStartOfSecond(timestamp) AS timestamp,
            start_tenor, end_tenor,
            argMax(bid_price, timestamp) AS bid,
            argMax(ask_price, timestamp) AS ask,
            argMax((bid_price + ask_price) / 2, timestamp) AS mid
        FROM ttmds.brn_v2
        WHERE toDate(timestamp) = '{asof_date}'
          AND type = 'S'
          AND bid_price > 0 AND ask_price > bid_price
          AND (start_tenor, end_tenor) IN ({pair_sql})
        GROUP BY timestamp, start_tenor, end_tenor
        ORDER BY start_tenor, end_tenor, timestamp
    """)
    if quotes.empty:
        return quotes
    quotes['timestamp'] = pd.to_datetime(quotes['timestamp'], utc=True).dt.tz_convert('Europe/London')
    quotes['start_tenor'] = pd.to_datetime(quotes['start_tenor']).dt.normalize()
    quotes['end_tenor'] = pd.to_datetime(quotes['end_tenor']).dt.normalize()
    return quotes


def asof_strip_quotes(trades: pd.DataFrame, quotes: pd.DataFrame) -> pd.DataFrame:
    empty = pd.DataFrame({'bid': np.nan, 'ask': np.nan, 'mid': np.nan}, index=trades.index)
    if trades.empty or quotes.empty:
        return empty

    left = trades[['timestamp', 'start_tenor', 'end_tenor']].copy()
    left['timestamp'] = pd.to_datetime(left['timestamp'], utc=True).dt.tz_convert('Europe/London').astype('datetime64[ns, Europe/London]')
    left['start_tenor'] = pd.to_datetime(left['start_tenor']).dt.normalize()
    left['end_tenor'] = pd.to_datetime(left['end_tenor']).dt.normalize()
    left = left.reset_index(drop=True)
    left['_i'] = np.arange(len(left))

    right = quotes.copy()
    right['timestamp'] = pd.to_datetime(right['timestamp'], utc=True).dt.tz_convert('Europe/London').astype('datetime64[ns, Europe/London]')

    pieces = []
    for (s, e), grp in left.groupby(['start_tenor', 'end_tenor'], sort=False):
        md = right[(right['start_tenor'] == s) & (right['end_tenor'] == e)]
        if md.empty:
            out = grp[['_i']].copy()
            out['bid'] = out['ask'] = out['mid'] = np.nan
            pieces.append(out)
            continue
        merged = pd.merge_asof(
            grp.sort_values('timestamp'),
            md[['timestamp', 'bid', 'ask', 'mid']].sort_values('timestamp'),
            on='timestamp',
            direction='backward',
        )
        pieces.append(merged[['_i', 'bid', 'ask', 'mid']])

    return pd.concat(pieces, ignore_index=True).set_index('_i').sort_index()[['bid', 'ask', 'mid']]


def lifecycle_decomposition(book_name: str, asof_date, verbose: bool = False) -> dict:
    p = get_spread_positions(book_name, HIST_START, asof_date)
    if p.empty:
        return {'book': book_name}

    p = p.groupby(level=0).last().sort_index()
    curves = spread_prices.groupby(level=0).last().sort_index()

    eod_p = p.groupby(p.index.date).tail(1)
    eod_p.index = eod_p.index.date
    prev_eod_pos = eod_p.shift().groupby(level=0).last()

    eod_c = curves.groupby(curves.index.date).tail(1)
    eod_c.index = eod_c.index.date
    prev_eod_c = eod_c.shift().groupby(level=0).last()

    bod_c = curves.groupby(curves.index.date).head(1)
    bod_c.index = bod_c.index.date
    bod_c = bod_c.groupby(level=0).last()

    day_curves = curves[curves.index.date == asof_date]
    day_pos = p[p.index.date == asof_date]
    if day_curves.empty or asof_date not in prev_eod_pos.index or asof_date not in bod_c.index:
        return {'book': book_name, 'error': 'missing BOD/EOD inputs'}

    xfer_events, trade_rows = [], []
    xfers = get_transfers(book_name, asof_date)
    if not xfers.empty:
        for _, row in xfers.iterrows():
            q = float(row['quantity'])
            q = q if str(row['side']).lower() == 'sell' else -q
            tenors = months_between(row['start_tenor'], row['end_tenor'])
            if tenors:
                xfer_events.append({'q': q, 'tenors': tenors, 'price': float(row['price']), 'ts': row['timestamp']})

    trades = get_trades(book_name, asof_date)
    pairs = []
    if not trades.empty:
        for _, row in trades.iterrows():
            q = float(row['quantity'])
            q = q if row['side'] == 'Buy' else -q
            try:
                start, end, tenors = parse_trade_tenors(row['instrument_key'])
            except Exception:
                continue
            if not tenors:
                continue
            pairs.append((start, end))
            trade_rows.append({
                'timestamp': row['timestamp'], 'start_tenor': start, 'end_tenor': end,
                'tenors': tenors, 'q': q, 'price': float(row['price']),
            })

    trade_df = pd.DataFrame(trade_rows)
    quotes = get_ttmds_spread_quotes(asof_date, pairs)
    if not trade_df.empty:
        qdf = asof_strip_quotes(trade_df, quotes)
        trade_df['ttmds_bid'] = qdf['bid'].values
        trade_df['ttmds_ask'] = qdf['ask'].values
        trade_df['ttmds_mid'] = qdf['mid'].values

    event_tenors = {t for e in xfer_events for t in e['tenors']}
    if not trade_df.empty:
        event_tenors |= {t for tenors in trade_df['tenors'] for t in tenors}

    Xt, X0 = day_curves.iloc[-1], bod_c.loc[asof_date]
    Q0, Qt = prev_eod_pos.loc[asof_date], day_pos.iloc[-1]
    cols = sorted(set(Q0.index) | set(Qt.index) | set(Xt.index) | set(X0.index) | event_tenors)
    Q0, Qt, Xt, X0 = [s.reindex(cols).fillna(0.0) for s in (Q0, Qt, Xt, X0)]

    if asof_date in prev_eod_c.index:
        Xprev = prev_eod_c.loc[asof_date].reindex(cols).fillna(0.0)
        overnight = float((CONTRACT_SIZE * Q0 * (X0 - Xprev)).sum())
    else:
        overnight = 0.0

    m2m = float((CONTRACT_SIZE * Qt * Xt).sum() - (CONTRACT_SIZE * Q0 * X0).sum())
    bod_inventory = float((CONTRACT_SIZE * Q0 * (Xt - X0)).sum())

    flow = pd.Series(0.0, index=cols)
    transfer_mark = trade_cf = 0.0
    for e in xfer_events:
        q, tenors, pj = e['q'], e['tenors'], e['price']
        n = len(tenors)
        trade_cf += -CONTRACT_SIZE * q * pj
        for t in tenors:
            flow.loc[t] += q
            transfer_mark += CONTRACT_SIZE * q * (float(Xt[t]) - pj / n)

    post_fill = mark_basis = mid_to_touch = touch_to_exec = 0.0
    if not trade_df.empty:
        for _, e in trade_df.iterrows():
            q, tenors, E = e['q'], e['tenors'], e['price']
            trade_cf += -CONTRACT_SIZE * q * E
            Xi = curve_at(e['timestamp'], tenors, curves).reindex(tenors).fillna(0.0)
            Xi_sum = float(Xi.sum())

            P, bid, ask = e['ttmds_mid'], e['ttmds_bid'], e['ttmds_ask']
            if pd.isna(P) or pd.isna(bid) or pd.isna(ask):
                P = T = Xi_sum
            else:
                T = float(ask) if q > 0 else float(bid)  # touch: ask if buy, bid if sell
                P = float(P)

            mark_basis += CONTRACT_SIZE * q * (Xi_sum - P)
            mid_to_touch += CONTRACT_SIZE * q * (P - T)
            touch_to_exec += CONTRACT_SIZE * q * (T - E)
            for t in tenors:
                flow.loc[t] += q
                post_fill += CONTRACT_SIZE * q * (float(Xt[t]) - float(Xi[t]))

    pure_exec = mid_to_touch + touch_to_exec
    pos_gap = Qt - (Q0 + flow)
    pos_gap_pnl = float((CONTRACT_SIZE * pos_gap * Xt).sum())
    market_risk = bod_inventory + transfer_mark + post_fill
    gross_decomp = overnight + market_risk + mark_basis + pure_exec
    company_gross = overnight + m2m + trade_cf

    out = {
        'book': book_name,
        'overnight': overnight,
        'm2m': m2m,
        'trade_cf': trade_cf,
        'company_gross': company_gross,
        'bod_inventory': bod_inventory,
        'transfer_mark': transfer_mark,
        'post_fill_mark': post_fill,
        'market_risk': market_risk,
        'internal_mark_basis': mark_basis,
        'mid_to_touch': mid_to_touch,
        'touch_to_exec': touch_to_exec,
        'pure_execution': pure_exec,
        'gross_decomp': gross_decomp,
        'pos_gap_pnl': pos_gap_pnl,
        'residual': company_gross - gross_decomp,
        'ttmds_fills_matched': int(trade_df['ttmds_mid'].notna().sum()) if not trade_df.empty else 0,
        'ttmds_fills_total': len(trade_df),
    }
    if verbose:
        print(pd.Series(out).drop(labels=['book']).round(2).to_string())
    return out

In [ ]:
# Single-book detail
decomp = lifecycle_decomposition(BOOK, DATE, verbose=True)

## All books — EOD summary

In [ ]:
summary = pd.DataFrame([lifecycle_decomposition(b, DATE) for b in BOOK_NAME_TO_ID]).set_index('book')
cols = [
    'overnight', 'company_gross',
    'bod_inventory', 'transfer_mark', 'post_fill_mark', 'market_risk',
    'internal_mark_basis', 'mid_to_touch', 'touch_to_exec', 'pure_execution',
    'gross_decomp', 'pos_gap_pnl', 'residual',
]
cols = [c for c in cols if c in summary.columns]
summary.loc['TEAM TOTAL', cols] = summary[cols].sum(numeric_only=True)
summary[cols].round(2)